# ModernFinBERT v2 Stage 2: Medium-Context Fine-Tuning

Load the Stage-1 model (trained on short-context data in NB 01a) and fine-tune on medium-context financial data.
- Stage-1 model: `neoyipeng/ModernFinBERT-v2-short`
- Dataset: `neoyipeng/modernfinbert-training-v2-medium` (29.5K train rows — earnings calls, SEC 10-K MD&A)
- MAX_LENGTH = 4096 (zero truncation — all passages fit within this window)
- Entity-aware input: `[CLS] entity [SEP] text [SEP]`
- Training target: `entity_sentiment`
- **Class-weighted loss** to handle POSITIVE skew (63% POSITIVE in training data)
- Evaluates on both medium AND short test sets to check for regression
- Pushes final model to `neoyipeng/ModernFinBERT-v2-medium` for Stage 3 to load

Estimated time on T4: ~2-3 hours

In [ ]:
%%capture
import os, re, torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer scikit-learn
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
%env UNSLOTH_DISABLE_FAST_GENERATION=1
%env TORCHDYNAMO_DISABLE=1

import torch
import torch._dynamo
torch._dynamo.config.disable = True

import numpy as np
from collections import Counter
from unsloth import FastModel, is_bfloat16_supported
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 3407
NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
LABEL_TO_ID = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
ID_TO_LABEL = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
MAX_LENGTH = 4096
STAGE1_MODEL = "neoyipeng/ModernFinBERT-v2-short"
HF_PUSH_NAME = "neoyipeng/ModernFinBERT-v2-medium"

cap = torch.cuda.get_device_capability()
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} not supported — need sm_70+ (T4/V100/A100)"
print(f"GPU: {torch.cuda.get_device_name()} (sm_{cap[0]}{cap[1]})")
print(f"Max tokens: {MAX_LENGTH}")

## 1. Load Medium-Context Data

In [ ]:
ds_med = load_dataset("neoyipeng/modernfinbert-training-v2-medium")
print("Medium-context dataset:")
print(ds_med)

for split in ds_med:
    ds_med[split] = ds_med[split].select_columns(["text", "entity", "entity_sentiment"])

def map_labels(examples):
    examples["label"] = [LABEL_TO_ID[l] for l in examples["entity_sentiment"]]
    return examples

for split in ds_med:
    ds_med[split] = ds_med[split].map(map_labels, batched=True)
    ds_med[split] = ds_med[split].remove_columns(["entity_sentiment"])

print(f"\nLabel distribution (train, entity_sentiment):")
labels = ds_med["train"]["label"]
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: {labels.count(i)}")

entities = ds_med["train"]["entity"]
n_with_entity = sum(1 for e in entities if e not in ("NONE", "MARKET", "None", "", None))
print(f"\nEntity coverage: {n_with_entity}/{len(entities)} ({100*n_with_entity/len(entities):.1f}%)")

In [ ]:
# Also load short-context TEST set for regression check
ds_short_test = load_dataset("neoyipeng/modernfinbert-training-v2", split="test")
ds_short_test = ds_short_test.select_columns(["text", "entity", "entity_sentiment"])
ds_short_test = ds_short_test.map(map_labels, batched=True)
ds_short_test = ds_short_test.remove_columns(["entity_sentiment"])
print(f"Short-context test set: {len(ds_short_test)} rows (for regression check)")

## 2. Load Stage-1 Model + Fresh LoRA

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=STAGE1_MODEL,
    auto_model=AutoModelForSequenceClassification,
    max_seq_length=MAX_LENGTH,
    dtype=None,
    num_labels=NUM_CLASSES,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    load_in_4bit=False,
)
print(f"Loaded stage-1 model from {STAGE1_MODEL}")

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    task_type="SEQ_CLS",
)

In [ ]:
# Entity-aware tokenization: sentence pair format [CLS] entity [SEP] text [SEP]
def tokenize_function(examples):
    entities = [
        e if e not in ("NONE", "MARKET", "None", "", None) else ""
        for e in examples["entity"]
    ]
    return tokenizer(entities, examples["text"], truncation=True, max_length=MAX_LENGTH)

# Tokenize medium-context train/val/test
tokenized_med = {}
for split in ds_med:
    tokenized_med[split] = ds_med[split].map(tokenize_function, batched=True, remove_columns=["text", "entity"])
    tokenized_med[split] = tokenized_med[split].rename_column("label", "labels")
    print(f"medium/{split}: {len(tokenized_med[split])} rows tokenized")

# Tokenize short-context test set (for regression check)
tokenized_short_test = ds_short_test.map(tokenize_function, batched=True, remove_columns=["text", "entity"])
tokenized_short_test = tokenized_short_test.rename_column("label", "labels")
print(f"short/test: {len(tokenized_short_test)} rows tokenized")

# Token length stats — verify zero truncation
lengths = [len(x) for x in tokenized_med["train"]["input_ids"]]
print(f"\nMedium-context token lengths (train):")
print(f"  min={min(lengths)}, median={sorted(lengths)[len(lengths)//2]}, "
      f"max={max(lengths)}, mean={np.mean(lengths):.0f}")
n_truncated = sum(1 for l in lengths if l >= MAX_LENGTH)
print(f"  Truncated to {MAX_LENGTH}: {n_truncated} rows ({100*n_truncated/len(lengths):.1f}%)")
assert n_truncated / len(lengths) < 0.01, f"Too many truncated: {n_truncated}"

## 3. Training

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

# Class-weighted loss to handle POSITIVE skew
label_counts = Counter(tokenized_med["train"]["labels"])
total = sum(label_counts.values())
class_weights = torch.tensor([
    total / (NUM_CLASSES * label_counts[i]) for i in range(NUM_CLASSES)
], dtype=torch.float32).to("cuda")
print(f"Label counts: {dict(label_counts)}")
print(f"Class weights: {class_weights}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits.view(-1, NUM_CLASSES), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Medium data: median ~2600 tokens, batch=16 fits on T4 with gradient checkpointing.
# Effective batch = 16 * 2 = 32.
# 1 epoch: 29.5K / 32 = ~921 steps, ~3-4h on T4.

trainer = WeightedTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tokenized_med["train"],
    eval_dataset=tokenized_med["validation"],
    compute_metrics=compute_metrics,
    args=TrainingArguments(
        output_dir="./modernfinbert-v2-medium-output",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        learning_rate=5e-5,
        weight_decay=0.001,
        warmup_ratio=0.06,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=SEED,
        group_by_length=True,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=False,
        report_to="none",
    ),
)

trainer.train()

## 4. Evaluation

In [ ]:
# Evaluate on medium-context test set
print("=" * 60)
print("MEDIUM-CONTEXT TEST SET")
print("=" * 60)
med_results = trainer.evaluate(tokenized_med["test"])
print(f"Accuracy={med_results['eval_accuracy']:.4f}, Macro F1={med_results['eval_macro_f1']:.4f}")

preds_med = trainer.predict(tokenized_med["test"])
y_pred_med = preds_med.predictions.argmax(axis=-1)
y_true_med = preds_med.label_ids
print(classification_report(y_true_med, y_pred_med, target_names=LABEL_NAMES))

# Evaluate on short-context test set (regression check)
print("=" * 60)
print("SHORT-CONTEXT TEST SET (regression check)")
print("=" * 60)
short_results = trainer.evaluate(tokenized_short_test)
print(f"Accuracy={short_results['eval_accuracy']:.4f}, Macro F1={short_results['eval_macro_f1']:.4f}")

preds_short = trainer.predict(tokenized_short_test)
y_pred_short = preds_short.predictions.argmax(axis=-1)
y_true_short = preds_short.label_ids
print(classification_report(y_true_short, y_pred_short, target_names=LABEL_NAMES))

print("\nTraining log:")
for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(f"  Epoch {entry.get('epoch', '?')}: loss={entry['eval_loss']:.4f}, "
              f"acc={entry.get('eval_accuracy', 0):.4f}, f1={entry.get('eval_macro_f1', 0):.4f}")

## 5. Push Final Model to HuggingFace

In [ ]:
merged = model.merge_and_unload()
merged.push_to_hub(HF_PUSH_NAME, private=False)
tokenizer.push_to_hub(HF_PUSH_NAME, private=False)
print(f"Final model pushed to {HF_PUSH_NAME}")
print(f"Class weights used: {class_weights}")